<a href="https://colab.research.google.com/github/ayushi777lodhi-stack/RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

if "COLAB_GPU" in os.environ:
  print("installing req")
  !pip install PyMuPDF
  !pip install tqdm
  !pip install accelerate
  !pip install bitsandbytes
  !pip install flash-attn --no-build-isolation

In [ ]:
!pip uninstall -y torch torchvision transformers sentence-transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers sentence-transformers

In [ ]:
from google.colab import files

uploaded=files.upload()

pdf_path=list(uploaded.keys())[0]
print("uploaded pdf:",pdf_path)



In [ ]:
import fitz
from tqdm.auto import tqdm

def text_formatting(text: str)->str:
  cleaned_text=text.replace("\n"," ").strip()
  return cleaned_text


def open_and_read_pdf(pdf_path:str)->list[dict]:
  doc=fitz.open(pdf_path)
  pages_and_texts=[]
  for page_number, page in tqdm(enumerate(doc)):
    text=page.get_text()
    text=text_formatting(text)
    pages_and_texts.append({"page number":page_number - 41,
                            "page_char_count":len(text),
                            "page_word_count":len(text.split(" ")),
                            "page_sentence_count_raw":len(text.split(". ")),
                            "page_token_count":len(text)/4,
                            "text":text})

    return pages_and_texts

pages_and_texts=open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

In [ ]:
import random
random.sample(pages_and_texts,k=3)

fixed chunking

In [ ]:
def chunk_text(text: str, chunk_size: int=500)-> list:
  chunks=[]
  current_chunk=''
  words.text.split()

  for word in words:
    if len(current_chunk)+len(word)+1<=chunk_size:
      current_chunk+=(word+' ')
    else:
      chunks.append(current_chunk.strip())
      current_chunk=word+' '


  if current_chunk:
    chunks.append(current_chunk.strip())

  return chunks



def chunk_pdf_pages(pages_and_texts: list,chunk_size: int=500)->list[dict]:
  all_chunks=[]
  for page in pages_and_texts:
    page_number=page["page_number"]
    page_text=page["text"]

    chunks=chunk_text(page_text, chunk_size=chunk_size)
    for i, chunk in enumerate(chunks):
      all_chunks.append({
          "page_number": page_number,
          "chunk_index": i,
          "chunk_char_count":len(chunk),
          "chunk_word_count":len(chunk.split()),
          "chunk_token_count":len(chunk)/4,
          "chunk_text":chunk
      })

  return all_chunks


  chunked_pages=chunk_pdf_pages(pages_and_texts,chunk_size=500)
  print(f"total chunks: "{len(chunked_pages)})
  print(f"first chunk(page{chunked_pages[0]['page_number']}):{chunked_pages[0]['chunk_text'][:200]}")

semantic chunking

In [ ]:
!pip -q install --upgrade "sentence-transformers==3.0.1" "transformers<5,>=4.41" scikit-learn nltk

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
nltk.download('punkt',quiet=True)

semantic_model=SentenceTransformer("all-MiniLM-L6-v2")

def semantic_chunk_text(text: str, similarity_threshold: float=0.8, max_tokens:int=500)->list:
  sentences=nltk.sent_tokenize(text)
  if not sentences:
    return []


  embeddings=semantic_model.encode(sentences)

  chunks=[]
  current_chunk=[sentences[0]]
  current_embedding=embeddings[0]

  for i in range(1,len(sentences)):
    sim=cosine_similarity([current_embedding],[embeddings[i]])[0][0]
    chunk_token_count=len(" ".join(current_chunk))//4

    if sim>=similarity_threshold and chunk_token_count<max_tokens:
      curent_chunk.append(sentences[i])
      current_emedding=np.mean([current_embedding,embeddings[i]],axis=0)
    else:
      chunks.append(" ".join(current_chunk))
      current_chunk=[sentences[i]]
      current_embedding=embeddings[i]

  if current_chunk:
    chunks.append(" ".join(current_chunk))

  return chunks


from tqdm.auto import tqdm

def semantic_chunk_pdf_pages(pages_and_texts:list,similarity_threshold:float=0.8,max_tokens:int =500)-> list[dict]:
  all_chunks=[]

  for page in tqdm(pages_and_texts):
    page_number=page["page_number"]
    page_text=page["text"]

    chunks=semantic_chunk_text(page_text,similarity_threshold=similarity_threshold, max_tokens=max_tokens)

    for i, chunk in enumerate(chunks):
      all_chunks.append({
          "page_number":page_number,
          "chunk_index":i,
          "chunk_char_count":len(chunk),
          "chunk_token_count":len(chunk.split()),
          "chunk_token_count":len(chunk)/4,
          "chunk_text":chunk
      })


  return all_chunks

In [ ]:
import nltk
nltk.download("punkt")

def recursive_chunk(text:str, max_chunk_size: int =1000, min_chunk_size:int=100)-> list:

  def split_chunk(chunk: str)-> list:
    if len(chunk) <= max_chunk_size:
      return [chunk]

    sections=chunk.split("\n\n")
    if len(sections)>1:
      result=[]
      for section in sections:
        if section.strip():
          result.extend(split_chunk(section.strip()))

      return result

    sections=chunk.split("\n")
    if len(sections)>1:
      result=[]
      for section in sections:
        if section.strip():
          result.extend(split_chunk(section.strip()))

      return result

    sentences=nltk.sent_tokenize(chunk)
    chunks,current_chunk, current_size=[],[],0

    for sentence in sentences:
      if current_size+len(sentence)>max_chunk_size:
        if current_chunk:
          chunks.append(" ".join(current_chunk))
        current_chunk=[sentence]
        current_size=len(sentence)

      else:
        current_chunk.append(sentence)
        current_size+=len(sentence)


    if current_chunk:
      chunks.append(" ".join(current_chunk))


    return chunks

  return split_chunk(text)



def recursive_chunk_pdf(pages_and_texts: list,max_chunk_size: int=1000, min_chunk_size: int=100)-> list[dict]:

  all_chunks=[]

  for page in tqdm(pages_and_texts):
    page_number=page["page_number"]
    page_text=page["text"]

    chunks=recursive_chunk(page_text, max_chunk_siz=max_chunk_size,min_chunk_size=min_chunk_size)
    for i, chunk in enumerate(chunks):
      all_chunks.append({
          "page_number": page_number,
          "chunk_index":i,
          "chunk_char_count":len(chunk),
          "chunk_token_count":len(chunk.split()),
          "chunk_token_count":len(chunk)/4,
          "chunk_text":chunk
      })


  return all_chunks

In [ ]:
from spcay.lang.en import English

nlp=English()

nlp.add_pipe("sentencizer")

doc=nlp("This is a sentence. This another sentence.")
assert len(list(doc.sents))==2

list(doc.sents)

In [ ]:
for item in tqdm(pages_and_texts):
  item["sentences"]=list(nlp(item["text"]).sents)

  item["sentences"]=[str(sentence) for sentence in item["sentences"]]

  item["page_sentence_count_spacy"]=len(item["sentences"])

In [ ]:
random.sample(pages_and_texts,k=1)

In [ ]:
num_sentence_chunk_size=10

def split_list(input_list:list, slice_size:int)->list[list[str]]:
  return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

for item in tqdm(pages_and_texts):
  item["sentence_chunks"]=split_list(input_list=item["sentences"], slice_size=slice_size)
  item["num_chunks"]=len(item["sentence_chunks"])

In [ ]:
random.sample(pages_and_texts,k=1)

In [ ]:
import re

pages_and_chunks=[]
for item in tqdm(pages_and_texts):
  for sentence_chunk in item["sentence_chunks"]:
    chunk_dict={}
    chunk_dict["page_number"]=item["page_number"]
    joined_sentence_chunk="".join(sentence_chunk).replace("  "," ").strip()
    joined_sentence_chunk=re.sub(r'\.([A-Z]',r'.\1',joined_sentence_chunk)
    chunk_dict["sentence_chunk"]=joined_sentence_chunk

    chunk_dict["chunk_char_count"]=len(joined_sentence_chunk)
    chunk_dict["chunk_token_count"]=len(word for word in joined_sentence_chunk.split(" "))
    chunk_dict["chunk_token_count"]=len(joined_sentence_chunk)/4

    pages_and_chunks.append(chunk_dict)

len(pages_and_chunks)


In [ ]:
random.sample(pages_and_chunks,k=1)

In [ ]:
df=pd.DataFrame(pages_and_texts)
df.describe().item()

In [ ]:
min_token_length=30
for row in df[df["chunk_token_count"]<=min_token_length].sample(5).iterrows():
  print(f"chunk token count: {row[1]["chunk_token_count"]} | text {row[1]["sentence_chunk"]}")

In [ ]:
pages_and_chunks_over_min_length=df[df["chunk_token_count"]]>min_token_length.to_dict(orient)
pages_and_chunks_over_min_length[:2]

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model=SentenceTransformer(model_name="all-mpnet-base-v2",
                                    device="cpu")
sentences=[
    "The Sentence Transformers library provides an easy and open-source way to create embeddings",
    "Sentences can be embedded one by one or as a list of strings.",
    "Embeddings are one of the most powerful concepts in machine learning",
    "Learn to use embeddings well and you'll be well on your way to being an AI engineer"
]

embeddings=embedding_model.encode(sentences)
embeddings_dict=dict(zip(sentences,embeddings))

for sentence, embedding in embeddings_dict.items():
  print("sentence",sentence)
  print("embedding",embedding)

In [ ]:
text_chunks=[item["sentence_chunk"] for item in pages_and_chunks_over_min_length]


In [ ]:
text_chunk_embeddings=embedding_model.encode(text_chunks, batch_size=32, convert_to_tensor=True)
print(text_chunk_embeddings)

In [ ]:
text_chunks_and_embeddings_df=pd.DataFrame(pages_and_chunks_over_min_length)
embeddings_df_save_path="text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path,index=False)


In [ ]:
text_chunks_and_embedding_df_load=pd.read_csv(embeddings_df_save_path)
text_chunks_and_embedding_df_load.head()

In [ ]:
import random
import torch
import numpy as npp
import pandas as pd

device="cuda" if torch.cuda.is_available() else "cpu"

text_chunks_and_embeddings_df=pd.read_csv("text_chunks_and_embeddings_df.csv")
text_chunks_and_embedding_df["embedding"]=text_chunks_and_embedding_df["embedding"].apply(lambda x:np.fromstring(x.strip("[]"),sep=" "))

pages_and_chunks=text_chunks_and_emeddings_df.to_dict(orient="records")

embeddings=torch.tensor(np.array(text_chunks_and_embedding_df["embedding"].tolist()),dtype=torch.float32).to(device)
print(embeddings.shape)

In [ ]:
text_chunks_and_embedding_df.head()

In [ ]:
embedding[0]

In [ ]:
from sentence_transformers import util, SentenceTransformer

embedding_model=SentenceTransformer(model_name="all-mpnet-base-v2",device=device)

In [ ]:
query="macronutrients functions"
print(f"query:{query}")

query_embedding=embedding_model.encode(query, convert_to_tensor=True)

from time import perf_counter as timer

dot_scores=util.dot_score(a=query_embedding,b=embeddings)[0]

top_results_dot_product=torch.topk(dot_scores,k=5)
print(top_results_dot_product)

In [ ]:
import textwrap

def print_wrappped(text,wrap_length=80):
  wrapped_text=textwrap.fill(text,wrap_length)
  print(wrapped_text)

In [ ]:
print(f"query {query}")
print("results:")
for score,idx in zip(top_results_dot_product[0],top_results_dot_product[1]):
  print(f"score:{score.4f}")
  print("text:")
  print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
  print(f"page number:{page_and_chunks[idx]["page_number"]}")
  print("\n")

In [ ]:
def retrieve(query,embeddings,model=embedding_model,n_resources_return):
  query_embedding=model.encode(query,convert_to_tensor=True)
  dot_scores=util.dot_score(query_embedding,embeddings)[0]

  scores,indices=torch.topk(input=dot_scores, k=n_resources_return)

  return scores,indices


def print_top_resulsts_and_scores(query,embeddings, pages_and_chunks=pages_and_chunks,n_resources_rerurn=5):
  scores,indices=retrieve(query=query,embeddings=embeddings,n_resources_return=n_resources_return)

print(f"query : {query}")
print("results:")
for score,idx in zip(scores,indices):
  print(f"score:{score.4f}")
  print("text:")
  print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
  print(f"page number:{page_and_chunks[idx]["page_number"]}")
  print("\n")

In [ ]:
query="symptoms of pellagra"
scores,indices=retrieve(query=query,embeddings=embeddings)
print(scores)
print(f"indices: {indices}")


In [ ]:
print_top_results_and_scores(query=query,embeddings=embeddings)

In [ ]:
def get_model_num_params(model: torch.nn.Module):
  returm sum([param.numel() for param in model.parameters])

get_model_num_params(llm_model)

In [ ]:
import torch
gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory
gpu_memory_gb=rand(gpu_memory_bytes/(2**30))
print(f"{gpu_memory_gb}")

In [ ]:
from huggingface_hub import login
login(token="hugging face yk")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import is_flash_attn_2_available
from transformers import BitsAndBytesConfig

qunatization_config=BitsAndBytesConfig(load_in_4bit=True,
                                       bnb_4bit_compute_dtype=torch.float16)

attn_implementation="sdpa"
model_id=model_id
tokenizer=AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_id)
llm_model=AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path,
                                               torch_dtype=torch.float16,
                                               quantization=quantization_config,
                                               low_cpu_mem_usage=False,
                                               attn_implementation=attn_implementation)


if not use_quantization_config:
  llm_model.to("cuda")

In [ ]:
input_text="what are the macronutrients, and what roles do they play in the human body?"
print(f"input:{input_text}")
prompt_template=[
    {"role":"user",
     "content":input_text}
]

prompt=tokenizer.apply_chat_template(conversation=prompt_template,
                                     tokenize=False,
                                     add_generation_prompt=True)

print(f"prompt formatted :{prompt}")

In [ ]:
input_ids=tokenizer(prompt, return_tensors="pt").to("cuda")
print(f"input ids{input_ids}")
outputs=llm.model.generate(**input_ids,
                           max_new_tokens=256)
print(f"model output :{outputs[0]}\n")

In [ ]:
outputs_decoded=tokenizer.decode(outputs[0])
print(f"decoded model output:{outputs_decoded}\n")

In [ ]:
print(f"input text:{input_text}")
print(f"output text: {outputs_decoded.replace(prompt,'').replace('<bos>','').replace('<eos>','')}")

In [ ]:
questions=[
    "How often should infants be breastfed?",
    "What are the symptoms of pellagra?",
    "What are macronutrients and what roles do they play in the human body?",
    "What is the RDI for protein per day?",
    "How does saliva help with digestion?",
    "water soluble vitamins"
]

query_list=questions

In [ ]:
def prompt_formatter(query,context_items):
  context="- "+"\n-".join([item["sentence_chunk"] for item in context_items])
  base_prompt="""based on the following context items, please answer the query.
  Give yourself room to think by extracting relevant passages from the context before answering the query.
  Make sure the answers are as explantory as possible.
  Use the following examples as refrence for the ideal answer style.
  \n Example 1:
  Query:What are fat-soluble vitamins?
  Answer: The fat-soluble vitamins include Vitamin A, Vitamin D, VItamin E, and VItamin K. These vitamins are absorbed along with the fats in the diet.
  \n Example 2:
  Query: What are the causes of type 2 diabetes?
  Answer: Type 2 diabetes is often associated with overnutrition, particularly the overconsumption of calories leading to obesity.
  \n Example 3:
  Query: What is the importance of hydration for physical performance?
  Answer: Hydration is crucial for physical perfprmance because water plays key roles iin maintaining blood volume, regulationg body temperature, and ensuring better health.
  \nNow use the following context items to answer the use query:
  {context}. The context contains the answer. Look for relevant information.
  \nRelevant passages: <extract relevant passages from the context here>
  User query:{query}
  Answer: """

  base_prompt=base_prompt.format(context=context, query=query)
  prompt_template=[
    {"role":"user",
     "content":input_text}
]

  prompt=tokenizer.apply_chat_template(conversation=prompt_template,
                                     tokenize=False,
                                     add_generation_prompt=True)

  return prompt



In [ ]:
import random
query=random.choice(query_list)
print(f"query:{query}")
scores,indices=retrieve_relevant_resources(query=query,
                                           embeddings=embeddings)
scores,indices

In [ ]:
import random
query=random.choice(query_list)
print(f"query:{query}")
scores,indices=retrieve_relevant_resources(query=query,
                                           embeddings=embeddings)
context_items=[pages_and_chunks[i] for i in indices]
prompt=prompt_formatter(query=query, context_items=context_items)
print(prompt)

In [ ]:
input_ids=tokenizer(prompt, return_tensors="pt").to("cuda")
outputs=llm.model.generate(**input_ids,
                           temperature=0.7,
                           do_sample=True,
                           max_new_tokens=256)

output_text=tokenizer.decode(outputs[0])
print(f"query:{query}")
print(f"answer: \n{output_text.replace(prompt,'')}")